# Phase 3 â€” Rule-Based Dispatch Baseline

## Objective

Build a deliberately simple dispatch baseline for the Lagos last-mile delivery scenario.

The baseline will:

1. Filter unavailable riders.
2. Assign deliveries using simple operational rules.
3. Respect basic rider capacity constraints.
4. Sequence assigned deliveries using greedy nearest-next-stop routing.
5. Use the Phase 2 road-network distance matrix instead of straight-line distance.
6. Simulate each rider's route.
7. Measure baseline operational performance.

This baseline is not intended to be optimal.

It establishes a measurable benchmark that later optimization and ML components must improve upon.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Get the project root (one level above the notebooks directory)
# In Jupyter, __file__ is not defined, so we work around it
import os
if '__file__' in dir():
    BASE_DIR = Path(__file__).parent.parent
else:
    # In Jupyter: assume we're in notebooks/ or running from project root
    BASE_DIR = Path.cwd()
    if BASE_DIR.name == 'notebooks':
        BASE_DIR = BASE_DIR.parent

DATA_RAW = BASE_DIR / "data" / "raw"
DATA_PROCESSED = BASE_DIR / "data" / "processed"

deliveries_raw = pd.read_csv(DATA_RAW / "deliveries.csv")
riders = pd.read_csv(DATA_RAW / "riders.csv")

distance_matrix = pd.read_csv(
    DATA_PROCESSED / "network_distance_matrix.csv",
    index_col=0
)

print("Deliveries:", deliveries_raw.shape)
print("Riders:", riders.shape)
print("Distance matrix:", distance_matrix.shape)

Deliveries: (60, 11)
Riders: (6, 9)
Distance matrix: (61, 61)


In [2]:
print("DELIVERY COLUMNS")
print(deliveries_raw.columns.tolist())

print("\nRIDER COLUMNS")
print(riders.columns.tolist())

display(deliveries_raw.head())
display(riders)

DELIVERY COLUMNS
['delivery_id', 'customer_id', 'area', 'latitude', 'longitude', 'weight_kg', 'package_size', 'priority', 'window_start', 'window_end', 'service_time_min']

RIDER COLUMNS
['rider_id', 'vehicle_type', 'available', 'shift_start', 'shift_end', 'break_start', 'break_end', 'capacity_kg', 'max_packages']


,delivery_id,customer_id,area,latitude,longitude,weight_kg,package_size,priority,window_start,window_end,service_time_min
0,D001,C001,Maryland,6.575974,3.365894,2.1,SMALL,NORMAL,14:00,18:00,15
1,D002,C002,Surulere,6.502082,3.365384,5.2,MEDIUM,NORMAL,12:00,16:00,5
2,D003,C003,Maryland,6.570127,3.365127,9.9,MEDIUM,NORMAL,12:00,16:00,10
3,D004,C004,Victoria Island,6.440734,3.428039,4.2,SMALL,HIGH,10:00,14:00,7
4,D005,C005,Yaba,6.512044,3.375040,5.7,MEDIUM,NORMAL,10:00,14:00,10


,rider_id,vehicle_type,available,shift_start,shift_end,break_start,break_end,capacity_kg,max_packages
0,R001,motorcycle,True,08:00,18:00,13:00,14:00,100,15
1,R002,motorcycle,True,08:00,18:00,13:00,14:00,100,15
2,R003,motorcycle,True,08:00,16:00,12:00,13:00,80,12
3,R004,motorcycle,True,10:00,19:00,14:00,15:00,100,15
4,R005,motorcycle,True,09:00,18:00,13:00,14:00,80,12
5,R006,motorcycle,False,08:00,18:00,12:00,13:00,100,15


In [3]:
deliveries = deliveries_raw.copy()
riders_work = riders.copy()

In [4]:
def time_to_minutes(time_value):
    """
    Convert HH:MM into minutes after midnight.

    Example:
    08:30 -> 510
    """
    hours, minutes = map(
        int,
        str(time_value).split(":")
    )

    return hours * 60 + minutes

In [5]:
available_riders = riders_work[
    riders_work["available"] == True
].copy()

print(
    f"Available riders: "
    f"{len(available_riders)} / {len(riders_work)}"
)

display(
    available_riders[
        [
            "rider_id",
            "shift_start",
            "shift_end",
            "capacity_kg",
            "max_packages"
        ]
    ]
)

Available riders: 5 / 6


,rider_id,shift_start,shift_end,capacity_kg,max_packages
0,R001,08:00,18:00,100,15
1,R002,08:00,18:00,100,15
2,R003,08:00,16:00,80,12
3,R004,10:00,19:00,100,15
4,R005,09:00,18:00,80,12


In [6]:
print(
    deliveries["priority"].value_counts(
        dropna=False
    )
)

display(
    deliveries[
        [
            "delivery_id",
            "area",
            "priority",
            "window_start",
            "window_end"
        ]
    ].head(15)
)

priority
NORMAL    28
HIGH      20
LOW       10
URGENT     2
Name: count, dtype: int64


,delivery_id,area,priority,window_start,window_end
0,D001,Maryland,NORMAL,14:00,18:00
1,D002,Surulere,NORMAL,12:00,16:00
2,D003,Maryland,NORMAL,12:00,16:00
3,D004,Victoria Island,HIGH,10:00,14:00
4,D005,Yaba,NORMAL,10:00,14:00
5,D006,Maryland,NORMAL,09:00,18:00
6,D007,Apapa,HIGH,10:00,14:00
7,D008,Lagos Island,NORMAL,09:00,12:00
8,D009,Lekki,HIGH,09:00,12:00
9,D010,Yaba,NORMAL,14:00,18:00


In [7]:
print("DELIVERY COLUMNS:")
print(deliveries.columns.tolist())

print("\nRIDER COLUMNS:")
print(riders_work.columns.tolist())

DELIVERY COLUMNS:
['delivery_id', 'customer_id', 'area', 'latitude', 'longitude', 'weight_kg', 'package_size', 'priority', 'window_start', 'window_end', 'service_time_min']

RIDER COLUMNS:
['rider_id', 'vehicle_type', 'available', 'shift_start', 'shift_end', 'break_start', 'break_end', 'capacity_kg', 'max_packages']


In [8]:
priority_rank = {
    "urgent": 0,
    "high": 1,
    "medium": 2,
    "normal": 3,
    "low": 4
}

deliveries["priority_rank"] = (
    deliveries["priority"]
    .str.lower()
    .map(priority_rank)
    .fillna(99)
)

deliveries = deliveries.sort_values(
    [
        "window_end",
        "priority_rank",
        "delivery_id"
    ]
).reset_index(drop=True)

In [9]:
def build_greedy_route(
    delivery_ids,
    distance_matrix,
    depot_id="DEP001"
):
    """
    Build a simple nearest-neighbour route.

    Starts at the depot, repeatedly visits the nearest
    unvisited assigned delivery, then returns to the depot.
    """

    remaining = set(delivery_ids)

    route = [depot_id]
    current = depot_id
    total_distance_m = 0.0

    while remaining:

        next_delivery = min(
            remaining,
            key=lambda delivery_id:
                distance_matrix.loc[
                    current,
                    delivery_id
                ]
        )

        leg_distance = distance_matrix.loc[
            current,
            next_delivery
        ]

        total_distance_m += leg_distance

        route.append(next_delivery)

        remaining.remove(next_delivery)

        current = next_delivery

    if len(route) > 1:

        return_distance = distance_matrix.loc[
            current,
            depot_id
        ]

        total_distance_m += return_distance

        route.append(depot_id)

    return route, total_distance_m

In [10]:
test_deliveries = [
    "D001",
    "D002",
    "D003",
    "D004"
]

test_route, test_distance = build_greedy_route(
    test_deliveries,
    distance_matrix
)

print("Route:")
print(" â†’ ".join(test_route))

print(
    f"\nTotal network distance: "
    f"{test_distance / 1000:.2f} km"
)

Route:
DEP001 â†’ D002 â†’ D003 â†’ D001 â†’ D004 â†’ DEP001

Total network distance: 52.76 km


In [29]:
rider_state = {}

for _, rider in available_riders.iterrows():

    rider_id = rider["rider_id"]

    rider_state[rider_id] = {
        "deliveries": [],
        "total_weight_kg": 0.0,
        "package_count": 0,
        "areas": set()
    }

rider_state

{'R001': {'deliveries': [],
  'total_weight_kg': 0.0,
  'package_count': 0,
  'areas': set()},
 'R002': {'deliveries': [],
  'total_weight_kg': 0.0,
  'package_count': 0,
  'areas': set()},
 'R003': {'deliveries': [],
  'total_weight_kg': 0.0,
  'package_count': 0,
  'areas': set()},
 'R004': {'deliveries': [],
  'total_weight_kg': 0.0,
  'package_count': 0,
  'areas': set()},
 'R005': {'deliveries': [],
  'total_weight_kg': 0.0,
  'package_count': 0,
  'areas': set()}}

In [31]:
assignments = []
unassigned = []

for _, delivery in deliveries.iterrows():

    delivery_id = delivery["delivery_id"]
    delivery_weight = float(delivery["weight_kg"])
    delivery_area = delivery["area"]

    feasible_riders = []

    for _, rider in available_riders.iterrows():

        rider_id = rider["rider_id"]
        state = rider_state[rider_id]

        # Check weight capacity
        new_weight = (
            state["total_weight_kg"]
            + delivery_weight
        )

        if new_weight > rider["capacity_kg"]:
            continue

        # Check package capacity
        new_package_count = (
            state["package_count"] + 1
        )

        if new_package_count > rider["max_packages"]:
            continue

        # Basic delivery-window / shift compatibility
        window_overlaps_shift = (
            delivery["window_end"]
            >= rider["shift_start"]
            and
            delivery["window_start"]
            <= rider["shift_end"]
        )

        if not window_overlaps_shift:
            continue

        # Prefer riders already serving this area
        same_area_penalty = (
            0
            if delivery_area in state["areas"]
            else 1
        )

        # Prefer rider with lower package utilization
        load_ratio = (
            new_package_count
            / rider["max_packages"]
        )

        score = (
            same_area_penalty,
            load_ratio
        )

        feasible_riders.append(
            (score, rider_id)
        )

    if not feasible_riders:

        unassigned.append(delivery_id)
        continue

    feasible_riders.sort(
        key=lambda x: x[0]
    )

    _, selected_rider = feasible_riders[0]

    state = rider_state[selected_rider]

    state["deliveries"].append(
        delivery_id
    )

    state["total_weight_kg"] += (
        delivery_weight
    )

    state["package_count"] += 1

    state["areas"].add(
        delivery_area
    )

    assignments.append(
        {
            "delivery_id": delivery_id,
            "rider_id": selected_rider,
            "area": delivery_area,
            "weight_kg": delivery_weight
        }
    )

SyntaxError: invalid syntax (3067352683.py, line 102)

In [ ]:
baseline_routes = []

for rider_id, state in rider_state.items():

    assigned_deliveries = (
        state["deliveries"]
    )

    if not assigned_deliveries:
        continue

    route, total_distance_m = (
        build_greedy_route(
            assigned_deliveries,
            distance_matrix,
            depot_id="DEP001"
        )
    )

    baseline_routes.append(
        {
            "rider_id": rider_id,
            "route": route,
            "delivery_count": len(
                assigned_deliveries
            ),
            "total_distance_m": (
                total_distance_m
            ),
            "total_distance_km": (
                total_distance_m / 1000
            )
        }
    )